# Ch08 — t 檢定與 z 檢定 (t-test & z-test)

**預計時間：** 2 小時

## 學習目標

1. 理解 **z 檢定** 與 **t 檢定** 的差異與適用條件
2. 掌握**前提假設檢驗**流程：常態性 (Normality) 與方差齊性 (Homogeneity of Variance)
3. 能執行**單樣本 t 檢定** (One-sample t-test)、**獨立雙樣本 t 檢定** (Independent two-sample t-test)、**成對 t 檢定** (Paired t-test)
4. 計算**效果量** (Effect Size, Cohen's d) 與**信賴區間** (Confidence Interval)
5. 將假設檢定流程應用於真實商業資料，做出可行的決策建議

## 前置知識

- **Ch07** — 假設檢定基礎 (Hypothesis Testing Basics)
- 基本描述統計量 (均值、標準差)
- Python 基礎 (pandas, numpy, scipy)

In [ ]:
# === 套件匯入 ===
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 6)

# 嘗試設定中文字型
try:
    plt.rcParams['font.family'] = ['Arial Unicode MS']  # macOS
except Exception:
    pass

print('All imports successful.')

In [ ]:
# === 載入資料集 ===

# 1. Titanic 資料集
titanic = pd.read_csv('case/train.csv')
print('Titanic shape:', titanic.shape)
display(titanic.head(3))

# 2. 模擬電商訂單資料 (ecommerce)
np.random.seed(42)
n_orders = 500
ecommerce = pd.DataFrame({
    'order_id': range(1, n_orders + 1),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n_orders, p=[0.3, 0.3, 0.2, 0.2]),
    'order_amount': np.concatenate([
        np.random.normal(120, 35, 150),   # North
        np.random.normal(105, 40, 150),   # South
        np.random.normal(115, 30, 100),   # East
        np.random.normal(110, 32, 100),   # West
    ]),
    'customer_age': np.random.randint(18, 65, n_orders),
    'is_returning': np.random.choice([0, 1], n_orders, p=[0.4, 0.6]),
})
ecommerce['order_amount'] = ecommerce['order_amount'].clip(lower=5).round(2)
print('\nEcommerce shape:', ecommerce.shape)
display(ecommerce.head(3))

---
## Part A — 前提假設檢驗 (Assumption Checks)

在執行 t 檢定或 z 檢定之前，必須確認資料是否符合檢定的前提假設。

### 檢定流程圖

```
資料 → 常態性檢驗 (Shapiro-Wilk)
         │
         ├── 通過 (p > 0.05) → 方差齊性 (Levene's test)
         │                        │
         │                        ├── 齊性 → Student's t-test
         │                        └── 不齊 → Welch's t-test
         │
         └── 不通過 (p ≤ 0.05) → 非參數檢定 (Mann-Whitney U)
```

### A.1 常態性檢驗 — Shapiro-Wilk Test

- **H₀：** 資料來自常態分佈
- **H₁：** 資料不來自常態分佈
- 若 p > 0.05，無法拒絕 H₀ → 認為資料近似常態

In [ ]:
# A.1 Shapiro-Wilk 常態性檢驗
age_data = titanic['Age'].dropna()

stat_sw, p_sw = stats.shapiro(age_data)
print(f'Shapiro-Wilk Test for Age:')
print(f'  W statistic = {stat_sw:.4f}')
print(f'  p-value     = {p_sw:.4f}')
print(f'  結論: {"資料近似常態" if p_sw > 0.05 else "資料偏離常態"}')

# 視覺化：直方圖 + Q-Q 圖
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(age_data, bins=30, edgecolor='black', alpha=0.7, density=True)
x_range = np.linspace(age_data.min(), age_data.max(), 100)
axes[0].plot(x_range, stats.norm.pdf(x_range, age_data.mean(), age_data.std()), 'r-', lw=2)
axes[0].set_title('Age Distribution with Normal Curve')
axes[0].set_xlabel('Age')

stats.probplot(age_data, plot=axes[1])
axes[1].set_title('Q-Q Plot of Age')

plt.tight_layout()
plt.show()

In [ ]:
# A.2 Levene's Test — 方差齊性檢驗
# 比較男女乘客的年齡方差是否一致

male_age = titanic.loc[titanic['Sex'] == 'male', 'Age'].dropna()
female_age = titanic.loc[titanic['Sex'] == 'female', 'Age'].dropna()

stat_lev, p_lev = stats.levene(male_age, female_age)
print(f"Levene's Test (Male vs Female Age):")
print(f'  W statistic = {stat_lev:.4f}')
print(f'  p-value     = {p_lev:.4f}')
print(f'  結論: {"方差齊性成立" if p_lev > 0.05 else "方差不齊，應使用 Welch t-test"}')
print(f'\n  Male  — mean={male_age.mean():.2f}, std={male_age.std():.2f}, n={len(male_age)}')
print(f'  Female — mean={female_age.mean():.2f}, std={female_age.std():.2f}, n={len(female_age)}')

### 前提假設口訣

| 步驟 | 檢驗 | 工具 | 判斷準則 |
|------|------|------|----------|
| 1 | 常態性 (Normality) | Shapiro-Wilk / K-S test | p > 0.05 → 常態 |
| 2 | 方差齊性 (Homoscedasticity) | Levene's test | p > 0.05 → 齊性 |
| 3 | 獨立性 (Independence) | 研究設計決定 | 觀測值互相獨立 |

> **口訣：** 先常態、後方差、再檢定。不常態就用無母數 (Non-parametric)。

In [ ]:
# A.3 整合檢驗函式 — 一鍵完成前提假設
def check_assumptions(group1, group2, name1='Group1', name2='Group2', alpha=0.05):
    """檢驗常態性與方差齊性，回傳建議的檢定方法"""
    results = {}
    
    # 常態性
    for name, data in [(name1, group1), (name2, group2)]:
        w, p = stats.shapiro(data)
        results[f'{name}_normal'] = p > alpha
        print(f'  Shapiro-Wilk ({name}): W={w:.4f}, p={p:.4f} -> {"Normal" if p > alpha else "Non-normal"}')
    
    # 方差齊性
    w, p = stats.levene(group1, group2)
    results['equal_var'] = p > alpha
    print(f'  Levene: W={w:.4f}, p={p:.4f} -> {"Equal variance" if p > alpha else "Unequal variance"}')
    
    # 建議
    both_normal = results[f'{name1}_normal'] and results[f'{name2}_normal']
    if both_normal and results['equal_var']:
        rec = "Student's t-test (equal_var=True)"
    elif both_normal:
        rec = "Welch's t-test (equal_var=False)"
    else:
        rec = 'Mann-Whitney U test (non-parametric)'
    
    print(f'\n  >>> Recommendation: {rec}')
    return rec, results

print('Helper function defined: check_assumptions(group1, group2, ...)')

---
## Part B — z 檢定 (z-test)

### 何時使用 z 檢定？

- **母體標準差 (σ) 已知**
- 樣本數 n ≥ 30（由中央極限定理保證）

### 公式

$$z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$$

| 符號 | 意義 |
|------|------|
| $\bar{x}$ | 樣本平均數 |
| $\mu_0$ | 虛無假設下的母體平均數 |
| $\sigma$ | 母體標準差（已知） |
| $n$ | 樣本數 |

### z vs t 的關鍵差異

| 比較項目 | z 檢定 | t 檢定 |
|----------|--------|--------|
| 母體 σ | 已知 | 未知（用 s 估計） |
| 分佈 | 標準常態 N(0,1) | t 分佈（自由度 df = n-1） |
| 樣本數 | 通常 n ≥ 30 | 任意（小樣本更優） |
| 實務使用 | 較少（σ 通常未知） | 較多 |

In [ ]:
# B.1 手動計算 z 檢定
# 假設情境：已知歷史資料顯示 Titanic 乘客年齡母體標準差 σ = 14.5
# H₀: μ = 30（乘客平均年齡為 30 歲）
# H₁: μ ≠ 30

sigma_known = 14.5       # 已知母體標準差
mu_0 = 30                # 虛無假設下的母體均值
alpha = 0.05

x_bar = age_data.mean()
n = len(age_data)

# z 統計量
z_stat = (x_bar - mu_0) / (sigma_known / np.sqrt(n))

# 雙尾 p-value
p_value_z = 2 * (1 - stats.norm.cdf(abs(z_stat)))

# 臨界值
z_critical = stats.norm.ppf(1 - alpha / 2)

print('=== z 檢定結果 ===')
print(f'樣本均值 x̄  = {x_bar:.2f}')
print(f'母體均值 μ₀ = {mu_0}')
print(f'母體標準差 σ = {sigma_known}')
print(f'樣本數 n    = {n}')
print(f'z statistic = {z_stat:.4f}')
print(f'p-value     = {p_value_z:.4f}')
print(f'z critical (α=0.05, two-tailed) = ±{z_critical:.4f}')
print(f'\n結論: {"拒絕 H₀ — 平均年齡顯著不等於 30" if p_value_z < alpha else "無法拒絕 H₀ — 無足夠證據認為平均年齡不等於 30"}')

In [ ]:
# B.2 z 檢定視覺化：拒絕域與 z 統計量
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(-4, 4, 300)
ax.plot(x, stats.norm.pdf(x), 'k-', lw=2, label='Standard Normal')

# 拒絕域 (Rejection Region)
x_left = x[x <= -z_critical]
x_right = x[x >= z_critical]
ax.fill_between(x_left, stats.norm.pdf(x_left), alpha=0.3, color='red', label='Rejection Region')
ax.fill_between(x_right, stats.norm.pdf(x_right), alpha=0.3, color='red')

# z 統計量位置
ax.axvline(z_stat, color='blue', linestyle='--', lw=2, label=f'z = {z_stat:.2f}')
ax.axvline(-z_critical, color='red', linestyle=':', alpha=0.7)
ax.axvline(z_critical, color='red', linestyle=':', alpha=0.7, label=f'Critical = +/-{z_critical:.2f}')

ax.set_title('z-test: Test Statistic vs Rejection Region')
ax.set_xlabel('z value')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

### z 檢定解讀

- z 統計量的絕對值越大，代表樣本均值與假設值的距離越遠（以標準誤為單位）
- 當 |z| > 1.96（雙尾 α = 0.05），拒絕虛無假設
- **注意：** 在實務中，母體標準差 σ 幾乎總是未知的，因此 **t 檢定** 更為常用

---
## Part C — 單樣本 t 檢定 (One-sample t-test)

### 問題：Titanic 乘客的平均年齡是否為 30 歲？

- **H₀：** μ = 30
- **H₁：** μ ≠ 30
- 母體標準差 σ **未知**，用樣本標準差 s 估計

### 公式

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}, \quad df = n - 1$$

In [ ]:
# C.1 單樣本 t 檢定
t_stat, p_value_t = stats.ttest_1samp(age_data, popmean=30)

print('=== 單樣本 t 檢定 (One-sample t-test) ===')
print(f'樣本均值 x̄ = {age_data.mean():.2f}')
print(f'樣本標準差 s = {age_data.std():.2f}')
print(f'樣本數 n = {len(age_data)}')
print(f'自由度 df = {len(age_data) - 1}')
print(f't statistic = {t_stat:.4f}')
print(f'p-value     = {p_value_t:.4f}')
print(f'\n結論: {"拒絕 H₀" if p_value_t < 0.05 else "無法拒絕 H₀"}')

### 完整解讀模板

報告 t 檢定結果時，應包含以下要素：

> 單樣本 t 檢定結果顯示，Titanic 乘客的平均年齡 (M = x̄, SD = s) 與假設值 30 歲之間  
> [存在 / 不存在] 統計顯著差異，t(df) = t_stat, p = p_value。

**關鍵要素：**
1. 描述統計量 (M, SD)
2. 檢定統計量 t(df)
3. p-value
4. 效果量 (Cohen's d)
5. 信賴區間 (95% CI)

In [ ]:
# C.2 計算 95% 信賴區間 (Confidence Interval)
from scipy.stats import t as t_dist

n_age = len(age_data)
mean_age = age_data.mean()
se_age = age_data.std(ddof=1) / np.sqrt(n_age)
df_age = n_age - 1

# 95% CI
t_crit = t_dist.ppf(0.975, df_age)
ci_lower = mean_age - t_crit * se_age
ci_upper = mean_age + t_crit * se_age

# Cohen's d (單樣本)
cohens_d_one = (mean_age - 30) / age_data.std(ddof=1)

print('=== 信賴區間 & 效果量 ===')
print(f'95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]')
print(f"Cohen's d = {cohens_d_one:.4f}")
print(f'\n解讀: 我們有 95% 的信心，母體平均年齡落在 [{ci_lower:.2f}, {ci_upper:.2f}] 之間')
print(f'       效果量 |d| = {abs(cohens_d_one):.2f} — {"大" if abs(cohens_d_one) >= 0.8 else "中" if abs(cohens_d_one) >= 0.5 else "小"} 效果')

In [ ]:
# C.3 信賴區間視覺化
fig, ax = plt.subplots(figsize=(10, 4))

ax.errorbar(mean_age, 0, xerr=t_crit * se_age, fmt='o', color='steelblue',
            capsize=10, capthick=2, linewidth=2, markersize=8)
ax.axvline(30, color='red', linestyle='--', lw=2, label=f'H0: mu = {mu_0}')
ax.set_yticks([])
ax.set_xlabel('Age')
ax.set_title(f'95% Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}]')
ax.legend()

# 標註
ax.annotate(f'x-bar = {mean_age:.2f}', (mean_age, 0.02), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

print(f'若 CI 包含 mu_0={mu_0}：無法拒絕 H0')
print(f'若 CI 不包含 mu_0={mu_0}：拒絕 H0')
print(f'本例：CI [{ci_lower:.2f}, {ci_upper:.2f}] {"包含" if ci_lower <= mu_0 <= ci_upper else "不包含"} {mu_0}')

---
## Part D — 獨立雙樣本 t 檢定 (Independent Two-sample t-test)

### 問題：Titanic 上男性與女性的平均年齡是否有顯著差異？

- **H₀：** μ_male = μ_female
- **H₁：** μ_male ≠ μ_female

### 兩種版本

| 版本 | 條件 | scipy 參數 |
|------|------|------------|
| Student's t | 方差齊性成立 | `equal_var=True` |
| Welch's t | 方差不齊 | `equal_var=False`（預設） |

In [ ]:
# D.1 分組 + 前提假設檢驗
print('=== 各組描述統計 ===')
for label, data in [('Male', male_age), ('Female', female_age)]:
    w, p = stats.shapiro(data)
    print(f'{label}: n={len(data)}, mean={data.mean():.2f}, std={data.std():.2f}, '
          f'Shapiro p={p:.4f} ({"常態" if p > 0.05 else "非常態"})')

print(f'\nLevene test: W={stat_lev:.4f}, p={p_lev:.4f}')
print(f'方差齊性: {"成立" if p_lev > 0.05 else "不成立"}')

# 決策
use_equal_var = p_lev > 0.05
print(f'\n→ 建議使用: {"Student t-test" if use_equal_var else "Welch t-test"}')

In [ ]:
# D.2 Student's t-test vs Welch's t-test

# Student's t (equal_var=True)
t_student, p_student = stats.ttest_ind(male_age, female_age, equal_var=True)
print("=== Student's t-test (equal_var=True) ===")
print(f't = {t_student:.4f}, p = {p_student:.4f}')

# Welch's t (equal_var=False)
t_welch, p_welch = stats.ttest_ind(male_age, female_age, equal_var=False)
print(f"\n=== Welch's t-test (equal_var=False) ===")
print(f't = {t_welch:.4f}, p = {p_welch:.4f}')

# 結論
chosen_p = p_student if use_equal_var else p_welch
chosen_name = "Student's" if use_equal_var else "Welch's"
print(f'\n採用 {chosen_name} t-test:')
print(f'結論: {"男女年齡有顯著差異" if chosen_p < 0.05 else "男女年齡無顯著差異"} (p = {chosen_p:.4f})')

In [ ]:
# D.3 Cohen's d — 效果量

def cohens_d(group1, group2):
    """計算獨立雙樣本的 Cohen's d (pooled SD 版本)"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std

d = cohens_d(male_age, female_age)
print(f"Cohen's d = {d:.4f}")
print(f'效果量大小: {"大 (large)" if abs(d) >= 0.8 else "中 (medium)" if abs(d) >= 0.5 else "小 (small)" if abs(d) >= 0.2 else "微弱 (negligible)"}')
print(f'\n效果量判讀標準 (Cohen, 1988):')
print(f'  |d| < 0.2  → 微弱')
print(f'  |d| ≥ 0.2  → 小效果')
print(f'  |d| ≥ 0.5  → 中效果')
print(f'  |d| ≥ 0.8  → 大效果')

### D.4 獨立雙樣本 t 檢定解讀

完整報告格式：

> 獨立樣本 t 檢定結果顯示，男性乘客 (M_male, SD_male) 與女性乘客 (M_female, SD_female) 的年齡  
> [存在 / 不存在] 統計顯著差異，t(df) = t_stat, p = p_value, Cohen's d = d。

**注意事項：**
- 統計顯著 ≠ 實務顯著，需同時報告效果量
- 大樣本下即使極小差異也可能 p < 0.05
- Welch's t-test 在多數情況下較為穩健，是更安全的預設選擇

### D.3b 非參數替代：Mann-Whitney U

當常態性假設不成立時，使用 **Mann-Whitney U 檢定**（又稱 Wilcoxon rank-sum test）：

- 不需要常態性假設
- 比較兩組的**秩次分佈** (rank distribution)
- H0: 兩組來自相同分佈

In [ ]:
# D.5 視覺化：男女年齡分佈比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱型圖
titanic.dropna(subset=['Age']).boxplot(column='Age', by='Sex', ax=axes[0])
axes[0].set_title('Age by Sex (Boxplot)')
axes[0].set_xlabel('Sex')
axes[0].set_ylabel('Age')
plt.sca(axes[0])
plt.title('Age by Sex (Boxplot)')

# 密度圖
male_age.plot.kde(ax=axes[1], label='Male', color='steelblue')
female_age.plot.kde(ax=axes[1], label='Female', color='coral')
axes[1].axvline(male_age.mean(), color='steelblue', linestyle='--', alpha=0.7)
axes[1].axvline(female_age.mean(), color='coral', linestyle='--', alpha=0.7)
axes[1].set_title('Age Density by Sex')
axes[1].set_xlabel('Age')
axes[1].legend()

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# D.3b Mann-Whitney U test（非參數替代）
u_stat, p_mann = stats.mannwhitneyu(male_age, female_age, alternative='two-sided')
print('=== Mann-Whitney U Test ===')
print(f'U statistic = {u_stat:.4f}')
print(f'p-value     = {p_mann:.4f}')
print(f'結論: {"有顯著差異" if p_mann < 0.05 else "無顯著差異"}')
print(f'\n比較: t-test p={chosen_p:.4f} vs Mann-Whitney p={p_mann:.4f}')
print('兩者結論一致' if (chosen_p < 0.05) == (p_mann < 0.05) else '兩者結論不一致，需進一步調查')

---
## Part E — 成對 t 檢定 (Paired t-test)

### 概念

當兩組觀測值**不獨立**（例如同一人的前後測量），使用成對 t 檢定。

- **典型場景：** 減重前後體重、訓練前後成績、A/B 測試中同一用戶的行為
- **原理：** 計算差值 d = X_after - X_before，對差值做單樣本 t 檢定 (H₀: μ_d = 0)

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

In [ ]:
# E.1 模擬減重資料 + 成對 t 檢定
np.random.seed(123)
n_subjects = 30

weight_before = np.random.normal(80, 10, n_subjects)
# 減重效果：平均減少 2.5 kg，個體差異大
weight_after = weight_before - np.random.normal(2.5, 3, n_subjects)

diet_df = pd.DataFrame({
    'subject_id': range(1, n_subjects + 1),
    'weight_before': weight_before.round(1),
    'weight_after': weight_after.round(1)
})
diet_df['diff'] = diet_df['weight_after'] - diet_df['weight_before']

print('=== 減重資料摘要 ===')
print(diet_df[['weight_before', 'weight_after', 'diff']].describe().round(2))

# 成對 t 檢定
t_paired, p_paired = stats.ttest_rel(diet_df['weight_after'], diet_df['weight_before'])

print(f'\n=== 成對 t 檢定 (Paired t-test) ===')
print(f't statistic = {t_paired:.4f}')
print(f'p-value     = {p_paired:.4f}')
print(f'平均差值    = {diet_df["diff"].mean():.2f} kg')
print(f'結論: {"減重計畫有顯著效果" if p_paired < 0.05 else "減重計畫效果不顯著"}')

# Cohen's d for paired
d_paired = diet_df['diff'].mean() / diet_df['diff'].std(ddof=1)
print(f"Cohen's d (paired) = {d_paired:.4f}")

In [ ]:
# E.2 視覺化成對資料
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 前後體重連線圖
for _, row in diet_df.iterrows():
    color = 'green' if row['diff'] < 0 else 'red'
    axes[0].plot([0, 1], [row['weight_before'], row['weight_after']], 
                 color=color, alpha=0.4, linewidth=1)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Before', 'After'])
axes[0].set_ylabel('Weight (kg)')
axes[0].set_title('Individual Weight Changes')

# 差值直方圖
axes[1].hist(diet_df['diff'], bins=10, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='No change')
axes[1].axvline(diet_df['diff'].mean(), color='green', linestyle='--', linewidth=2, label='Mean diff')
axes[1].set_xlabel('Weight Change (kg)')
axes[1].set_title('Distribution of Weight Changes')
axes[1].legend()

# 箱型圖比較
axes[2].boxplot([diet_df['weight_before'], diet_df['weight_after']], 
                labels=['Before', 'After'])
axes[2].set_ylabel('Weight (kg)')
axes[2].set_title('Before vs After (Boxplot)')

plt.tight_layout()
plt.show()

### E.3 成對 t 檢定解讀

- 成對 t 檢定利用了「同一受試者」的資訊，**消除個體差異**帶來的變異
- 相比獨立樣本 t 檢定，成對 t 檢定通常有**更高的統計檢定力** (Power)
- 前提假設：**差值** (d = after - before) 需要近似常態分佈

> **常見錯誤：** 將成對資料當作獨立樣本來分析，會低估檢定力，可能遺漏真正的效果。

---
## Part F — 電商資料實戰 (E-commerce Case Study)

### 問題：北區 (North) 與南區 (South) 的訂單金額是否有顯著差異？

我們將執行完整的假設檢定 pipeline：
1. 探索性資料分析 (EDA)
2. 常態性檢驗
3. 方差齊性檢驗
4. 選擇並執行適當的 t 檢定
5. 計算效果量與信賴區間
6. 做出商業決策建議

In [ ]:
# E.2b Wilcoxon signed-rank test（成對 t 的非參數替代）
w_wilcox, p_wilcox = stats.wilcoxon(diet_df['weight_after'], diet_df['weight_before'])
print('=== Wilcoxon Signed-Rank Test ===')
print(f'W statistic = {w_wilcox:.4f}')
print(f'p-value     = {p_wilcox:.4f}')
print(f'結論: {"有顯著差異" if p_wilcox < 0.05 else "無顯著差異"}')
print(f'\n比較: Paired t p={p_paired:.4f} vs Wilcoxon p={p_wilcox:.4f}')

In [ ]:
# F.1 完整 pipeline

north = ecommerce.loc[ecommerce['region'] == 'North', 'order_amount']
south = ecommerce.loc[ecommerce['region'] == 'South', 'order_amount']

print('=' * 60)
print('Step 1: 探索性資料分析 (EDA)')
print('=' * 60)
for label, data in [('North', north), ('South', south)]:
    print(f'  {label}: n={len(data)}, mean={data.mean():.2f}, '
          f'std={data.std():.2f}, median={data.median():.2f}')

print(f'\n  差異 (North - South): {north.mean() - south.mean():.2f}')

print(f'\n{"=" * 60}')
print('Step 2: 常態性檢驗 (Shapiro-Wilk)')
print('=' * 60)
for label, data in [('North', north), ('South', south)]:
    w, p = stats.shapiro(data)
    result = 'PASS (p > 0.05)' if p > 0.05 else 'FAIL (p <= 0.05)'
    print(f'  {label}: W={w:.4f}, p={p:.4f} → {result}')

print(f'\n{"=" * 60}')
print('Step 3: 方差齊性檢驗 (Levene)')
print('=' * 60)
w_lev, p_lev_ec = stats.levene(north, south)
eq_var = p_lev_ec > 0.05
print(f'  W={w_lev:.4f}, p={p_lev_ec:.4f}')
print(f'  方差齊性: {"成立" if eq_var else "不成立"}')

print(f'\n{"=" * 60}')
print(f'Step 4: {"Student" if eq_var else "Welch"}\'s t-test')
print('=' * 60)
t_ec, p_ec = stats.ttest_ind(north, south, equal_var=eq_var)
print(f'  t statistic = {t_ec:.4f}')
print(f'  p-value     = {p_ec:.4f}')
print(f'  結論: {"有顯著差異" if p_ec < 0.05 else "無顯著差異"}')

print(f'\n{"=" * 60}')
print('Step 5: 效果量 & 信賴區間')
print('=' * 60)
d_ec = cohens_d(north, south)
diff_mean = north.mean() - south.mean()
se_diff = np.sqrt(north.var(ddof=1)/len(north) + south.var(ddof=1)/len(south))
ci_diff = (diff_mean - 1.96 * se_diff, diff_mean + 1.96 * se_diff)
print(f"  Cohen's d = {d_ec:.4f}")
print(f'  95% CI for difference: [{ci_diff[0]:.2f}, {ci_diff[1]:.2f}]')

In [ ]:
# F.2 視覺化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Boxplot
ns_data = ecommerce[ecommerce['region'].isin(['North', 'South'])]
sns.boxplot(data=ns_data, x='region', y='order_amount', ax=axes[0], palette='Set2')
axes[0].set_title('Order Amount: North vs South')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Order Amount ($)')

# Violin plot
sns.violinplot(data=ns_data, x='region', y='order_amount', ax=axes[1], palette='Set2', inner='quartile')
axes[1].set_title('Distribution Shape (Violin)')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Order Amount ($)')

# KDE overlay
north.plot.kde(ax=axes[2], label='North', color='steelblue')
south.plot.kde(ax=axes[2], label='South', color='coral')
axes[2].axvline(north.mean(), color='steelblue', linestyle='--', alpha=0.7)
axes[2].axvline(south.mean(), color='coral', linestyle='--', alpha=0.7)
axes[2].set_title('KDE: North vs South')
axes[2].set_xlabel('Order Amount ($)')
axes[2].legend()

plt.tight_layout()
plt.show()

### F.3 商業決策建議

根據分析結果，我們可以做出以下建議：

1. **若差異顯著 (p < 0.05)：**
   - 調查造成區域差異的原因（產品組合、行銷策略、消費者結構）
   - 效果量若大 → 值得投入資源調整策略
   - 效果量若小 → 差異可能不具實務意義

2. **若差異不顯著 (p >= 0.05)：**
   - 兩區域可沿用相同行銷策略
   - 但仍需關注其他指標（轉換率、客單價趨勢）

> **重要：** 統計顯著 ≠ 實務顯著。務必結合效果量 (Cohen's d) 和信賴區間來做最終判斷。

In [ ]:
# F.4 進階：多區域比較概覽
print('=== 四大區域訂單金額概覽 ===')
summary = ecommerce.groupby('region')['order_amount'].agg(['count', 'mean', 'std', 'median']).round(2)
summary.columns = ['n', 'Mean', 'Std', 'Median']
display(summary)

# 所有區域箱型圖
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=ecommerce, x='region', y='order_amount', palette='Set2', 
            order=['North', 'South', 'East', 'West'], ax=ax)
ax.set_title('Order Amount by Region')
ax.set_xlabel('Region')
ax.set_ylabel('Order Amount ($)')
plt.tight_layout()
plt.show()

print('\n提示: 若要比較三個以上群組，應使用 ANOVA（見 Ch03）而非多次 t 檢定，')
print('      以避免多重比較問題 (Multiple Comparison Problem) 導致 Type I Error 膨脹。')

---
## 速查表 (Cheat Sheet)

| 檢定方法 | 適用場景 | scipy 函式 | 前提條件 |
|----------|----------|------------|----------|
| **z 檢定** | 母體 σ 已知，n ≥ 30 | 手動計算 | 常態 or CLT |
| **單樣本 t** | 一組 vs 已知值 | `ttest_1samp()` | 常態 |
| **獨立 t (Student)** | 兩組獨立，方差齊 | `ttest_ind(equal_var=True)` | 常態 + 方差齊 |
| **獨立 t (Welch)** | 兩組獨立，方差不齊 | `ttest_ind(equal_var=False)` | 常態 |
| **成對 t** | 同一對象前後測 | `ttest_rel()` | 差值常態 |
| **Mann-Whitney U** | 非常態替代 | `mannwhitneyu()` | 無 |
| **Wilcoxon** | 非常態成對替代 | `wilcoxon()` | 無 |

## 重點整理

1. **先檢驗前提再做 t 檢定**：常態性 (Shapiro-Wilk) → 方差齊性 (Levene) → 選擇檢定
2. **z 檢定 vs t 檢定**：母體 σ 已知用 z，未知用 t（實務中幾乎都用 t）
3. **三種 t 檢定的選擇：**
   - 一組 vs 常數 → 單樣本 t
   - 兩組獨立 → 獨立雙樣本 t (Student/Welch)
   - 同一對象前後 → 成對 t
4. **報告要完整**：描述統計 + t(df) + p-value + Cohen's d + 95% CI
5. **統計顯著 ≠ 實務顯著**：大樣本下微小差異也會顯著，要看效果量

## 練習題

### 練習 1：Titanic 票價分析
檢驗存活者 (Survived=1) 與未存活者 (Survived=0) 的票價 (Fare) 是否有顯著差異。
- 執行完整的前提假設檢驗
- 選擇適當的 t 檢定
- 報告效果量與信賴區間

### 練習 2：艙等與年齡
比較一等艙 (Pclass=1) 與三等艙 (Pclass=3) 乘客的平均年齡差異。

### 練習 3：電商回購分析
檢驗回購客戶 (is_returning=1) 與新客戶 (is_returning=0) 的訂單金額是否有顯著差異。

### 練習 4（挑戰）：檢定力分析
若要偵測 Cohen's d = 0.5 的效果量，在 α = 0.05、power = 0.8 的條件下，
每組需要多少樣本？（提示：使用 `statsmodels.stats.power`）

### F.5 進階練習：回購客戶 vs 新客戶

In [ ]:
# TODO: 練習 1 — 存活者 vs 未存活者 票價分析
# survived_fare = titanic.loc[titanic['Survived'] == 1, 'Fare']
# not_survived_fare = titanic.loc[titanic['Survived'] == 0, 'Fare']
# Step 1: Shapiro-Wilk
# Step 2: Levene
# Step 3: t-test
# Step 4: Cohen's d
# Step 5: 95% CI

In [ ]:
# F.5 回購 vs 新客戶訂單金額比較
returning = ecommerce.loc[ecommerce['is_returning'] == 1, 'order_amount']
new_cust = ecommerce.loc[ecommerce['is_returning'] == 0, 'order_amount']

print('=== 回購 vs 新客戶 ===')
rec, _ = check_assumptions(returning, new_cust, 'Returning', 'New')

t_ret, p_ret = stats.ttest_ind(returning, new_cust, equal_var=False)
d_ret = cohens_d(returning, new_cust)
print(f'\nWelch t-test: t={t_ret:.4f}, p={p_ret:.4f}')
print(f"Cohen's d = {d_ret:.4f}")
print(f'結論: {"回購與新客戶訂單金額有顯著差異" if p_ret < 0.05 else "回購與新客戶訂單金額無顯著差異"}')

## 導覽列

| 章節 | 主題 |
|------|------|
| ← Ch07 | 假設檢定基礎 |
| **Ch08** | **t 檢定與 z 檢定 (本章)** |
| Ch09 → | 進階假設檢定 |

---
*Ch08 — t 檢定與 z 檢定 | iSpan Python 資料分析*

---
## 補充：Type I & Type II Error 回顧

| | H₀ 為真 | H₀ 為假 |
|------|---------|--------|
| **拒絕 H₀** | Type I Error (alpha) | Correct (Power = 1-beta) |
| **不拒絕 H₀** | Correct | Type II Error (beta) |

- **alpha (alpha)：** 犯 Type I Error 的機率（通常設為 0.05）
- **Power (1 - beta)：** 正確拒絕錯誤 H0 的能力（目標 >= 0.80）
- 增加樣本數可以提高 Power
- 效果量越大，Power 越高

### 延伸閱讀

- Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.)
- Welch, B. L. (1947). The generalization of Student's problem when several different population variances are involved. *Biometrika*, 34(1-2), 28-35.
- [scipy.stats documentation](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [Real Statistics: t-test](https://www.real-statistics.com/students-t-distribution/)

In [ ]:
# 補充：檢定力分析 (Power Analysis)
try:
    from statsmodels.stats.power import TTestIndPower
    
    power_analysis = TTestIndPower()
    
    # 給定 effect size, alpha, power → 求 n
    for effect_size in [0.2, 0.5, 0.8]:
        n_required = power_analysis.solve_power(
            effect_size=effect_size, alpha=0.05, power=0.8, alternative='two-sided'
        )
        print(f"Cohen's d = {effect_size} → 每組需要 n = {int(np.ceil(n_required))}")
    
    # Power curve
    fig, ax = plt.subplots(figsize=(10, 6))
    sample_sizes = np.arange(10, 300)
    for es in [0.2, 0.5, 0.8]:
        powers = [power_analysis.power(es, n, 0.05) for n in sample_sizes]
        ax.plot(sample_sizes, powers, label=f'd = {es}')
    ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Power = 0.80')
    ax.set_xlabel('Sample Size (per group)')
    ax.set_ylabel('Power')
    ax.set_title('Power Curve by Effect Size')
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('statsmodels not installed. Run: pip install statsmodels')

In [ ]:
# TODO: 練習 3 — 電商回購客戶分析
# (提示：使用 F.5 區塊的結果進行擴展分析)
# 1. 分組比較描述統計
# 2. 視覺化分佈
# 3. 假設檢定
# 4. 撰寫商業建議

In [ ]:
# TODO: 練習 2 — 一等艙 vs 三等艙年齡比較
# pclass1_age = titanic.loc[titanic['Pclass'] == 1, 'Age'].dropna()
# pclass3_age = titanic.loc[titanic['Pclass'] == 3, 'Age'].dropna()
# rec, _ = check_assumptions(pclass1_age, pclass3_age, 'Pclass1', 'Pclass3')
# t_val, p_val = stats.ttest_ind(pclass1_age, pclass3_age, equal_var=False)
# d_val = cohens_d(pclass1_age, pclass3_age)
# print(f't={t_val:.4f}, p={p_val:.4f}, d={d_val:.4f}')